<a href="https://colab.research.google.com/github/Rharrabirihab/Pr-diction_Panne_Komax/blob/main/encodage_komax_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import RidgeCV, LassoCV


In [11]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os
import pickle

# ── 1. CHARGER SPLITS ──────────────────────────
# Update the path to look for the file in Google Drive
file_path = '/content/drive/MyDrive/Projet_Maintenance_Komax/splits_raw.pkl'

with open(file_path, 'rb') as f:
    splits_raw = pickle.load(f)

print("✅ Splits chargés")
for name, s in splits_raw.items():
    print(f"  {name} : Train={s['Xtr'].shape}")

✅ Splits chargés
  Type_Of_Failure : Train=(14658, 6)
  Serial_Device : Train=(14658, 7)
  Down_Time : Train=(14658, 9)
  When_Panne : Train=(14558, 13)


In [13]:
# ── 2. COLONNES À ENCODER ──────────────────────
CAT_COLS = [
    'Machine', 'Type Of Failure', 'Serial Device',
    'Defect Code', 'Shift', 'Equipe',
    'Last_Type_Failure', 'Last_Defect_Code',
    'Last_Serial_Device', 'Last_Shift'
]

In [14]:
# ── 3. LABEL ENCODING ──────────────────────────
label_encoders = {}
label_mappings = {}
splits_encoded = {}

for name, s in splits_raw.items():
    print(f"\n--- Encodage {name} ---")

    Xtr = s['Xtr'].copy()
    Xte = s['Xte'].copy()
    ytr = s['ytr'].copy()
    yte = s['yte'].copy()

    # Encoder chaque colonne catégorielle
    for col in CAT_COLS:
        if col not in Xtr.columns:
            continue

        le = LabelEncoder()

        # FIT sur train uniquement
        Xtr[col + '_enc'] = le.fit_transform(
            Xtr[col].fillna('INCONNU').astype(str))

        # TRANSFORM sur test
        Xte[col + '_enc'] = Xte[col].fillna(
            'INCONNU').astype(str).apply(
            lambda x: x if x in le.classes_
            else le.classes_[0])
        Xte[col + '_enc'] = le.transform(Xte[col + '_enc'])

        # Sauvegarder mapping
        mapping = dict(zip(
            le.classes_,
            le.transform(le.classes_)))
        label_encoders[f"{name}_{col}"] = le
        label_mappings[f"{name}_{col}"]  = mapping

        print(f"  ✅ {col}_enc "
              f"({len(le.classes_)} classes)")

        # Afficher mapping
        print(f"     Mapping (5 premiers) :")
        for k, v in list(mapping.items())[:5]:
            print(f"       '{k}' → {v}")

        # Supprimer colonne originale
        Xtr = Xtr.drop(columns=[col])
        Xte = Xte.drop(columns=[col])

    # Encoder target si classification
    task = ('reg' if name in ['Down_Time', 'When_Panne']
            else 'clf')

    if task == 'clf':
        le_target = LabelEncoder()
        ytr_enc   = le_target.fit_transform(
            ytr.fillna('INCONNU').astype(str))
        yte_enc   = yte.fillna('INCONNU').astype(str).apply(
            lambda x: x if x in le_target.classes_
            else le_target.classes_[0])
        yte_enc   = le_target.transform(yte_enc)

        label_encoders[f"{name}_target"] = le_target
        label_mappings[f"{name}_target"] = dict(zip(
            le_target.classes_,
            le_target.transform(le_target.classes_)))

        print(f"\n  ✅ Target encodé : "
              f"{len(le_target.classes_)} classes")
        print(f"     Mapping target :")
        for k, v in list(
                label_mappings[f"{name}_target"].items())[:5]:
            print(f"       '{k}' → {v}")

        splits_encoded[name] = {
            'Xtr'   : Xtr,
            'Xte'   : Xte,
            'ytr'   : ytr_enc,
            'yte'   : yte_enc,
            'feats' : list(Xtr.columns),
            'task'  : task,
        }
    else:
        splits_encoded[name] = {
            'Xtr'   : Xtr,
            'Xte'   : Xte,
            'ytr'   : ytr.values,
            'yte'   : yte.values,
            'feats' : list(Xtr.columns),
            'task'  : task,
        }




--- Encodage Type_Of_Failure ---
  ✅ Machine_enc (63 classes)
     Mapping (5 premiers) :
       'KOMAX_1' → 0
       'KOMAX_10' → 1
       'KOMAX_11' → 2
       'KOMAX_12' → 3
       'KOMAX_13' → 4
  ✅ Defect Code_enc (167 classes)
     Mapping (5 premiers) :
       'BC01' → 0
       'BC02' → 1
       'BC03' → 2
       'BC04' → 3
       'BC05' → 4
  ✅ Shift_enc (3 classes)
     Mapping (5 premiers) :
       'Après-midi' → 0
       'Matin' → 1
       'Nuit' → 2
  ✅ Equipe_enc (3 classes)
     Mapping (5 premiers) :
       'Equipe_1' → 0
       'Equipe_2' → 1
       'Equipe_3' → 2

  ✅ Target encodé : 18 classes
     Mapping target :
       'BLOC COUTEAUX' → 0
       'CONVOYEUR' → 1
       'DEMARRAGE PARC' → 2
       'ELECT/PNEUM' → 3
       'IT (SOFT/HARDWARE)' → 4

--- Encodage Serial_Device ---
  ✅ Machine_enc (63 classes)
     Mapping (5 premiers) :
       'KOMAX_1' → 0
       'KOMAX_10' → 1
       'KOMAX_11' → 2
       'KOMAX_12' → 3
       'KOMAX_13' → 4
  ✅ Type Of Failure_enc (

In [15]:
# ── 4. ONE HOT ENCODING (optionnel) ────────────
print("\n--- One Hot Encoding ---")
LOW_CARD = ['Shift', 'Equipe']

for name, s in splits_raw.items():
    Xtr = s['Xtr'].copy()
    Xte = s['Xte'].copy()

    cols_ohe = [c for c in LOW_CARD if c in Xtr.columns]
    if not cols_ohe:
        continue

    ohe = OneHotEncoder(
        sparse_output=False,
        handle_unknown='ignore')

    ohe_tr = ohe.fit_transform(Xtr[cols_ohe])
    ohe_te = ohe.transform(Xte[cols_ohe])

    ohe_cols  = ohe.get_feature_names_out(cols_ohe)
    df_ohe_tr = pd.DataFrame(
        ohe_tr, columns=ohe_cols,
        index=Xtr.index)
    df_ohe_te = pd.DataFrame(
        ohe_te, columns=ohe_cols,
        index=Xte.index)

    # Ajouter au dataset
    splits_encoded[name]['Xtr_ohe'] = pd.concat(
        [splits_encoded[name]['Xtr'], df_ohe_tr], axis=1)
    splits_encoded[name]['Xte_ohe'] = pd.concat(
        [splits_encoded[name]['Xte'], df_ohe_te], axis=1)

    print(f"  ✅ OHE {name} : {list(ohe_cols)}")




--- One Hot Encoding ---
  ✅ OHE Type_Of_Failure : ['Shift_Après-midi', 'Shift_Matin', 'Shift_Nuit', 'Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']
  ✅ OHE Serial_Device : ['Shift_Après-midi', 'Shift_Matin', 'Shift_Nuit', 'Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']
  ✅ OHE Down_Time : ['Shift_Après-midi', 'Shift_Matin', 'Shift_Nuit', 'Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']
  ✅ OHE When_Panne : ['Equipe_Equipe_1', 'Equipe_Equipe_2', 'Equipe_Equipe_3']


In [16]:
# ── 4b. STANDARDISATION pour tâches de régression ─────────
# Ridge et Lasso nécessitent des features sur la même échelle.
# On standardise uniquement les splits de régression
# (Down Time et When Panne) sans toucher aux splits de classification.
#
# ⚠️ FIT sur train uniquement → transform sur test
#    (même logique anti-leakage que pour LabelEncoder)

from sklearn.preprocessing import StandardScaler
import numpy as np

REGRESSION_TASKS = ['downtime', 'whenpanne']  # adapte aux noms de tes splits

scalers = {}

for name, s in splits_encoded.items():
    if s.get('task') not in REGRESSION_TASKS:
        continue

    print(f"\n--- Standardisation {name} ---")

    Xtr = s['Xtr'].copy()
    Xte = s['Xte'].copy()

    # Colonnes numériques à standardiser (exclure colonnes encodées _enc)
    NUM_COLS = [c for c in Xtr.columns
                if Xtr[c].dtype in [np.float64, np.int64, float, int]
                and '_enc' not in c]

    sc = StandardScaler()
    Xtr[NUM_COLS] = sc.fit_transform(Xtr[NUM_COLS])   # FIT sur train
    Xte[NUM_COLS] = sc.transform(Xte[NUM_COLS])        # TRANSFORM sur test

    splits_encoded[name]['Xtr_scaled'] = Xtr
    splits_encoded[name]['Xte_scaled'] = Xte
    scalers[name] = sc

    print(f"  Features standardisées : {NUM_COLS}")
    print(f"  Xtr_scaled : {Xtr.shape}")

# Sauvegarder les scalers (utiles pour Streamlit)
import pickle
with open('scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print("\n✅ Standardisation OK")
print("✅ scalers.pkl sauvegardé")



✅ Standardisation OK
✅ scalers.pkl sauvegardé


In [17]:
# ── 5. SAUVEGARDER ─────────────────────────────
with open('splits_encoded.pkl', 'wb') as f:
    pickle.dump(splits_encoded, f)
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open('label_mappings.pkl', 'wb') as f:
    pickle.dump(label_mappings, f)

print("\n" + "="*45)
print("  RÉSUMÉ ENCODAGE")
print("="*45)
for name, s in splits_encoded.items():
    print(f"\n  {name} ({s['task']}) :")
    print(f"    Xtr : {s['Xtr'].shape}")
    print(f"    Xte : {s['Xte'].shape}")
    print(f"    Features : {s['feats']}")

print("\n✅ splits_encoded.pkl sauvegardé")
print("✅ label_encoders.pkl sauvegardé")
print("✅ label_mappings.pkl sauvegardé")


  RÉSUMÉ ENCODAGE

  Type_Of_Failure (clf) :
    Xtr : (14658, 6)
    Xte : (3665, 6)
    Features : ['Hour', 'Reaction Time (min)', 'Machine_enc', 'Defect Code_enc', 'Shift_enc', 'Equipe_enc']

  Serial_Device (clf) :
    Xtr : (14658, 7)
    Xte : (3665, 7)
    Features : ['Hour', 'Reaction Time (min)', 'Machine_enc', 'Type Of Failure_enc', 'Defect Code_enc', 'Shift_enc', 'Equipe_enc']

  Down_Time (reg) :
    Xtr : (14658, 9)
    Xte : (3665, 9)
    Features : ['Hour', 'Reaction Time (min)', 'Waiting Time (min)', 'Machine_enc', 'Type Of Failure_enc', 'Serial Device_enc', 'Defect Code_enc', 'Shift_enc', 'Equipe_enc']

  When_Panne (reg) :
    Xtr : (14558, 13)
    Xte : (3640, 13)
    Features : ['Last_Hour', 'Last_Down_Time', 'Last_Reaction_Time', 'Last_Waiting_Time', 'Nb_Pannes_Avant', 'Mean_Down_Time', 'Mean_Time_Between', 'Machine_enc', 'Equipe_enc', 'Last_Type_Failure_enc', 'Last_Defect_Code_enc', 'Last_Serial_Device_enc', 'Last_Shift_enc']

✅ splits_encoded.pkl sauvegardé
✅ la

In [18]:
Xtr.head()

,Machine,Last_Type_Failure,Last_Defect_Code,Last_Serial_Device,Last_Hour,Last_Shift,Last_Down_Time,Last_Reaction_Time,Last_Waiting_Time,Nb_Pannes_Avant,Mean_Down_Time,Mean_Time_Between,Equipe
12312,KOMAX_48,IT (SOFT/HARDWARE),IT06,AUTRE,23,Nuit,12.67,0.15,3.83,22,10.411364,49.413320,Equipe_1
15720,KOMAX_57,KIT-JOINT,SAP12,23314,15,Après-midi,11.15,0.07,2.85,344,13.109012,23.909762,Equipe_1
2425,KOMAX_19,IT (SOFT/HARDWARE),IT10,AUTRE,5,Nuit,11.93,0.08,9.43,89,11.745618,18.403198,Equipe_3
1437,KOMAX_15,MINI-APPLICATEUR,MAP01,LS52777,10,Matin,16.85,0.28,8.97,28,12.993929,36.819465,Equipe_2
1519,KOMAX_15,MINI-APPLICATEUR,MAP01,LS90550,13,Matin,8.57,0.30,3.28,110,11.809091,40.855726,Equipe_2


In [19]:
# ── 6. class_weight & Ridge / Lasso ────────────────────────
# Cette cellule ne remplace PAS ton notebook de modélisation.
# Elle centralise les helpers à importer dans ce notebook.
#
#  A) class_weight='balanced'  → classification (Type Of Failure, Serial Device)
#  B) compute_sample_weight    → XGBoost (ne supporte pas class_weight directement)
#  C) RidgeCV / LassoCV        → régression (Down Time, When Panne)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, classification_report,
                              r2_score, mean_absolute_error)
import pickle

with open('splits_encoded.pkl', 'rb') as f:
    splits_encoded = pickle.load(f)
with open('scalers.pkl', 'rb') as f:
    scalers = pickle.load(f)


# ════════════════════════════════════════════════════════════
#  A) CLASSIFICATION — class_weight='balanced'
#     À utiliser dans ton notebook de modélisation comme ceci :
# ════════════════════════════════════════════════════════════

def make_rf_classifier(n_estimators=300):
    """Random Forest avec class_weight balancé (anti-déséquilibre)."""
    return RandomForestClassifier(
        n_estimators=n_estimators,
        class_weight='balanced',   # ← le paramètre clé
        random_state=42,
        n_jobs=-1
    )

def xgb_sample_weights(y_train):
    """Pour XGBoost qui ne supporte pas class_weight directement."""
    return compute_sample_weight('balanced', y_train)

print("✅ Helpers classification prêts (make_rf_classifier, xgb_sample_weights)")


# ════════════════════════════════════════════════════════════
#  B) RÉGRESSION — Ridge + Lasso sur Down Time / When Panne
# ════════════════════════════════════════════════════════════

REGRESSION_TASKS = ['downtime', 'whenpanne']
results_reg = {}

for name, s in splits_encoded.items():
    if s.get('task') not in REGRESSION_TASKS:
        continue

    Xtr = s.get('Xtr_scaled', s['Xtr'])
    Xte = s.get('Xte_scaled', s['Xte'])
    ytr = s['ytr']
    yte = s['yte']

    print(f"\n{'='*50}")
    print(f"  Régression : {name}")
    print(f"{'='*50}")

    # Ridge
    ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
    ridge.fit(Xtr, ytr)
    pred_r = ridge.predict(Xte)
    r2_r  = r2_score(yte, pred_r)
    mae_r = mean_absolute_error(yte, pred_r)
    print(f"  Ridge  (alpha={ridge.alpha_:.2f}) — R²: {r2_r:.3f}  MAE: {mae_r:.1f} min")

    # Lasso
    lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso.fit(Xtr, ytr)
    pred_l = lasso.predict(Xte)
    r2_l  = r2_score(yte, pred_l)
    mae_l = mean_absolute_error(yte, pred_l)
    print(f"  Lasso  (alpha={lasso.alpha_:.4f}) — R²: {r2_l:.3f}  MAE: {mae_l:.1f} min")

    import pandas as pd
    coefs = pd.Series(lasso.coef_, index=Xtr.columns if hasattr(Xtr, 'columns') else range(Xtr.shape[1]))
    n_kept = (coefs.abs() > 0).sum()
    print(f"  Lasso  features gardées : {n_kept}/{len(coefs)}")
    if n_kept > 0:
        print(coefs[coefs.abs() > 0].sort_values(key=abs, ascending=False).head(5).to_string())

    results_reg[name] = {
        'ridge_r2': r2_r, 'ridge_mae': mae_r, 'ridge_alpha': ridge.alpha_,
        'lasso_r2': r2_l, 'lasso_mae': mae_l, 'lasso_alpha': lasso.alpha_,
        'lasso_coefs': coefs
    }

print("\n✅ Ridge + Lasso terminés")


✅ Helpers classification prêts (make_rf_classifier, xgb_sample_weights)

✅ Ridge + Lasso terminés
